# Heart Disease Analytics — Predictive Modelling

This notebook provides a reproducible classification workflow for the UCI Heart Disease Cleveland dataset.

**Models:** Majority baseline, Logistic Regression, Decision Tree, Random Forest  
**Metrics:** Accuracy, Precision, Recall, F1, Specificity, ROC-AUC, PR-AUC

> Performance values are calculated from the loaded dataset; no model results are hard-coded.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

from src.data_loader import load_heart_disease
from src.preprocessing import build_preprocessor, make_train_test_data
from src.modeling import build_models, evaluate_models


In [ ]:
df = load_heart_disease()
print("Dataset shape:", df.shape)
display(df.head())


## 1. Define the Prediction Target

For binary classification:

- `0` → no disease
- `1–4` → disease present


In [ ]:
X, y = make_train_test_data(df)

print("Feature shape:", X.shape)
print("Target distribution:")
display(y.value_counts().sort_index())


## 2. Stratified Train/Test Split

The test set is held out before model fitting. Stratification preserves the target-class proportions as closely as possible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


In [ ]:
preprocessor = build_preprocessor(X_train)
models = build_models(preprocessor)

results, fitted_models = evaluate_models(
    models, X_train, X_test, y_train, y_test
)

display(results.sort_values("ROC_AUC", ascending=False))


## 3. Confusion Matrices


In [ ]:
for name, model in fitted_models.items():
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()


## 4. ROC Curves


In [ ]:
for name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name)

plt.title("ROC Curves")
plt.tight_layout()
plt.show()


## 5. Metric Interpretation

For disease as the positive class:

- **Precision** = TP / (TP + FP)
- **Recall / Sensitivity** = TP / (TP + FN)
- **Specificity** = TN / (TN + FP)
- **F1** = harmonic mean of precision and recall

Model selection should not rely on accuracy alone. The operational consequences of false positives and false negatives should be considered with healthcare stakeholders.


## 6. Reproducibility Checklist

- [ ] Dataset source/version recorded
- [ ] Target mapping recorded
- [ ] Random seed recorded
- [ ] Train/test split recorded
- [ ] Preprocessing fitted on training data only
- [ ] Metrics reproduced from code
- [ ] Limitations documented
- [ ] No unsupported clinical claims
